# ENGG2112: Complete Data Engineering and A-to-B Handoff

**Task:** classify APTOS retinal photographs into the original five grades, 0-4.
**Authoring assistance:** Codex; student review and understanding remain required.
**Notebook prepared:** 14 October 2026.

Use this as the repository's single maintained Notebook entry point. It contains the
Python function definitions, executable workflow, recorded data findings, report
notes and handoff instructions. The `src` modules remain available for command-line
use and tests; this Notebook runs without importing them.

**Status: PROVISIONAL.** Similar-image review, team agreement on model inputs, and
the final split freeze remain incomplete. There are no model performance results.
File verification does not establish clinical label accuracy or patient independence.

Read and execute cells from top to bottom after selecting the correct Python kernel.
The published Notebook deliberately contains no saved execution outputs or retinal
images. Numerical results below are explicitly identified as recorded results.

## Contents
1. Environment, paths and execution modes
2. Shared helpers and five-grade labels
3. Data integrity and duplicate audit
4. Class distribution and audit findings
5. Group-aware provisional splitting
6. Deterministic image preparation
7. Verified loading and handoff to B
8. Recorded data analysis and report contribution
9. Acceptance checklist, limitations and AI-use record

## 1. Environment, paths and execution modes

Install the repository requirements in your Python environment before selecting it
as the Notebook kernel. From the project root, for example:

```text
python -m pip install -r requirements.txt
```

For a standalone copy of this Notebook, the required packages are `numpy`, `pandas`,
`Pillow`, `matplotlib`, `scikit-learn`, `ipykernel` and `nbformat`. The project was
checked with Python 3.12; exact versions are listed in `requirements-lock.txt`.

Choose one mode in the configuration cell:

| Mode | Use it when | Required local files | Behaviour |
|---|---|---|---|
| `inspect` (default) | A's prepared project or the private handoff is available | `data/train.csv`, split files, prepared images and their metadata | Reads existing data and checks it; does not regenerate or overwrite the dataset |
| `build` | Starting with authorised Kaggle originals | `data/train.csv`, `data/train_images/*.png` | Writes a new self-contained handoff under `artifacts/notebook_runs/RUN_NAME/` |

Set `PROJECT_ROOT` to your own project directory if the kernel starts elsewhere.
A fresh GitHub clone contains neither the original images nor the prepared handoff.
The three CSV files already at the repository root do not provide the images; put
the authorised training CSV at `data/train.csv` and images at `data/train_images/`.

Build mode refuses an existing run directory. To build again, choose a new
`RUN_NAME`; do not overwrite the shared provisional handoff. Original images remain
unchanged. A full audit decodes every original image and may take several minutes.

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
import platform
import re
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, __version__ as pillow_version
from sklearn.model_selection import GroupShuffleSplit
from IPython.display import display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__, '| NumPy:', np.__version__, '| Pillow:', pillow_version)

In [ ]:
# Edit these settings before running the remaining cells.
PROJECT_ROOT = Path.cwd().resolve()
# Example: PROJECT_ROOT = Path(r'/path/to/your/project')
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

MODE = 'inspect'  # 'inspect' or 'build'
RUN_NAME = 'notebook_run_001'  # Use a new name for every build.
SEED = 2112
WORKERS = 4
NEAR_DISTANCE = 2
REVIEWED_GROUPS = None  # Optional Path to a documented id_code,reviewed_group CSV.
SHOW_PRIVATE_PREVIEW = False

if MODE not in {'inspect', 'build'}:
    raise ValueError("MODE must be 'inspect' or 'build'.")
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('RUN_NAME must contain only letters, digits, underscore or hyphen.')
ROOT = PROJECT_ROOT
LABELS_PATH = ROOT / 'data/train.csv'
RAW_IMAGES = ROOT / 'data/train_images'
WORK_ROOT = ROOT if MODE == 'inspect' else ROOT / 'artifacts/notebook_runs' / RUN_NAME
AUDIT_DIR = WORK_ROOT / 'artifacts/audit'
SPLIT_DIR = WORK_ROOT / 'artifacts/splits'
PREPARED_DIR = WORK_ROOT / 'data/processed_224_v1'

if not LABELS_PATH.is_file():
    raise FileNotFoundError(f'Place the authorised training CSV at {LABELS_PATH}.')
if MODE == 'build':
    if not RAW_IMAGES.is_dir():
        raise FileNotFoundError(f'Extract authorised originals into {RAW_IMAGES}.')
    if WORK_ROOT.exists():
        raise FileExistsError(f'{WORK_ROOT} exists. Choose a new RUN_NAME to preserve it.')
    if REVIEWED_GROUPS is not None and not Path(REVIEWED_GROUPS).is_file():
        raise FileNotFoundError('The configured reviewed-groups CSV does not exist.')
else:
    required = [SPLIT_DIR / 'split_manifest.csv', SPLIT_DIR / 'split_summary.json',
                PREPARED_DIR / 'processed_manifest.csv', PREPARED_DIR / 'preprocessing_config.json']
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError('Inspect mode requires the prepared handoff. Extract it here, '
                                "or use MODE='build' with original data. Missing: " + ', '.join(missing))

print('Mode:', MODE)
print('Source project:', ROOT)
print('Working handoff:', WORK_ROOT)

## 2. Shared helpers and five-grade labels

The five labels remain unchanged: 0 No DR, 1 Mild, 2 Moderate, 3 Severe,
4 Proliferative DR. Missing/unsafe IDs, repeated IDs and invalid labels cause an
error rather than silent row removal. A dHash match is only a review candidate.

In [ ]:
CLASS_NAMES = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative DR'}

In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')


def read_labels(path):
    frame = pd.read_csv(path, dtype={'id_code': 'string'})
    missing = {'id_code', 'diagnosis'} - set(frame.columns)
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    ids = frame['id_code']
    if ids.isna().any() or not ids.str.fullmatch(r'[A-Za-z0-9_-]+').all():
        raise ValueError('Missing or unsafe image ID. Review labels before proceeding.')
    if ids.duplicated().any():
        raise ValueError('Duplicate image IDs in CSV. Resolve explicitly; no rows were dropped.')
    numeric = pd.to_numeric(frame['diagnosis'], errors='coerce')
    if not numeric.isin(range(5)).all():
        raise ValueError('Labels must be integers 0, 1, 2, 3, 4; no binary remapping is used.')
    frame['diagnosis'] = numeric.astype(int)
    return frame


def difference_hash(image):
    """64-bit dHash: only a review heuristic, NOT a patient identifier."""
    from PIL import Image
    small = np.asarray(image.convert('L').resize((9, 8), Image.Resampling.LANCZOS))
    bits = (small[:, 1:] > small[:, :-1]).ravel()
    return f'{int.from_bytes(np.packbits(bits).tobytes(), "big"):016x}'

In [ ]:
labels = read_labels(LABELS_PATH)
print('Label rows:', len(labels))
print('Columns:', list(labels.columns))
print('Labels SHA-256:', file_hash(LABELS_PATH))

## 3. Data integrity and duplicate audit

The following cells define the complete audit implementation. Images are verified
and fully decoded. SHA-256 of RGB pixels plus dimensions identifies exact matches;
files with identical pixels but conflicting labels are provisionally quarantined.
The dHash search flags potential similar images for manual review. It does not
identify patients, correct labels or prove that unrelated images are independent.

The plot helper also writes a **private** example sheet locally. Do not publish
that sheet or saved Notebook outputs containing source images.

In [ ]:
def inspect_image(row):
    image_id, label, path = row
    result = {'id_code': image_id, 'diagnosis': label, 'path': str(path),
              'status': 'ok', 'error': ''}
    if not path.is_file():
        return {**result, 'status': 'missing', 'error': 'Image file not found'}
    try:
        result['bytes'] = path.stat().st_size
        result['file_sha256'] = file_hash(path)
        with warnings.catch_warnings():
            warnings.simplefilter('error', Image.DecompressionBombWarning)
            with Image.open(path) as source:
                source.verify()  # verify PNG integrity, including CRCs
            with Image.open(path) as source:
                source.load()  # force full decoding rather than just reading the header
                result.update(width=source.width, height=source.height, mode=source.mode)
                rgb = source.convert('RGB')
                result['pixel_sha256'] = hashlib.sha256(
                    f'{rgb.width}x{rgb.height}:RGB:'.encode() + rgb.tobytes()).hexdigest()
                result['dhash'] = difference_hash(rgb)
                thumb = np.asarray(rgb.resize((64, 64))).astype(float)
                result['mean_intensity'] = float(thumb.mean())
                result['intensity_std'] = float(thumb.std())
                result['dark_pixel_fraction'] = float((thumb.max(axis=2) <= 10).mean())
    except Exception as error:
        result.update(status='unreadable', error=f'{type(error).__name__}: {error}')
    return result


def find_near_duplicates(frame, max_distance=2):
    """List visually similar candidates for human review. Do not change labels."""
    records = frame.to_dict('records')
    found = []
    for i, left in enumerate(records):
        left_hash = int(left['dhash'], 16)
        for right in records[i + 1:]:
            if left['pixel_sha256'] == right['pixel_sha256']:
                continue
            distance = (left_hash ^ int(right['dhash'], 16)).bit_count()
            if distance <= max_distance:
                found.append({'left_id': left['id_code'], 'right_id': right['id_code'],
                              'left_label': left['diagnosis'], 'right_label': right['diagnosis'],
                              'dhash_distance': distance, 'review_decision': ''})
    columns = ['left_id', 'right_id', 'left_label', 'right_label', 'dhash_distance', 'review_decision']
    return pd.DataFrame(found, columns=columns).sort_values('dhash_distance')

In [ ]:
def make_plots(labels, images, destination):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    counts = labels.diagnosis.value_counts().reindex(range(5), fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.bar([f'{k}: {CLASS_NAMES[k]}' for k in range(5)], counts, color='#237a91')
    ax.bar_label(bars, padding=3)
    ax.set(ylabel='Number of labelled images', title='APTOS 2019: original five-grade labels')
    ax.tick_params(axis='x', labelsize=8)
    ax.set_ylim(0, max(counts) * 1.14)
    fig.tight_layout()
    fig.savefig(destination / 'class_distribution.png', dpi=170)
    plt.close(fig)
    good = images[images.status.eq('ok')]
    if not good.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(good.width, good.height, s=8, alpha=.3)
        ax.set(xlabel='Width (pixels)', ylabel='Height (pixels)', title='Original image dimensions')
        fig.tight_layout()
        fig.savefig(destination / 'image_dimensions.png', dpi=170)
        plt.close(fig)
        sheet = Image.new('RGB', (3 * 220, 5 * 250), 'white')
        draw = ImageDraw.Draw(sheet)
        for label in range(5):
            for col, row in enumerate(good[good.diagnosis.eq(label)].head(3).itertuples()):
                with Image.open(row.path) as source:
                    sample = ImageOps.pad(source.convert('RGB'), (210, 210))
                x, y = col * 220, label * 250
                sheet.paste(sample, (x, y))
                draw.text((x + 3, y + 214), f'{label}: {CLASS_NAMES[label]}', fill='black')
                draw.text((x + 3, y + 229), row.id_code, fill='black')
        sheet.save(destination / 'PRIVATE_sample_images.jpg', quality=90)


def run_audit(labels_path, image_dir, output_dir, workers=4, near_distance=2):
    output_dir.mkdir(parents=True, exist_ok=True)
    labels = read_labels(labels_path)
    jobs = [(r.id_code, int(r.diagnosis), image_dir / f'{r.id_code}.png')
            for r in labels.itertuples()]
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for number, result in enumerate(pool.map(inspect_image, jobs), 1):
            results.append(result)
            if number % 250 == 0 or number == len(jobs):
                print(f'Checked {number}/{len(jobs)} images', flush=True)
    images = pd.DataFrame(results)
    for column in ['pixel_sha256', 'dhash']:
        if column not in images:
            images[column] = ''
    images['duplicate_group'] = images['pixel_sha256']
    good = images[images.status.eq('ok')].copy()
    exact = good[good.pixel_sha256.duplicated(keep=False)].copy()
    conflicts = set(good.groupby('pixel_sha256').diagnosis.nunique().loc[lambda s: s > 1].index)
    images['label_conflict'] = images.pixel_sha256.isin(conflicts)
    images['eligible'] = images.status.eq('ok') & ~images.label_conflict
    images.to_csv(output_dir / 'image_inventory.csv', index=False)
    exact.to_csv(output_dir / 'exact_duplicates.csv', index=False)
    images[images.label_conflict].to_csv(output_dir / 'conflicting_labels.csv', index=False)
    near = find_near_duplicates(good, near_distance)
    near.to_csv(output_dir / 'near_duplicate_candidates.csv', index=False)
    expected = {f'{i}.png' for i in labels.id_code}
    present = {p.name for p in image_dir.glob('*.png')}
    counts = labels.diagnosis.value_counts().reindex(range(5), fill_value=0)
    class_table = pd.DataFrame({'grade': range(5), 'name': list(CLASS_NAMES.values()),
                               'count': counts.values, 'percent': (counts.values / len(labels) * 100)})
    class_table.to_csv(output_dir / 'class_distribution.csv', index=False)
    summary = {'label_rows': len(labels), 'label_columns': list(labels.columns),
               'labels_sha256': file_hash(labels_path), 'image_directory': str(image_dir.resolve()),
               'readable_images': len(good), 'missing_images': int(images.status.eq('missing').sum()),
               'unreadable_images': int(images.status.eq('unreadable').sum()),
               'extra_png_files': sorted(present - expected),
               'class_counts': {str(k): int(v) for k, v in counts.items()},
               'exact_duplicate_groups': int(exact.pixel_sha256.nunique()),
               'images_in_exact_duplicate_groups': len(exact),
               'conflicting_label_groups': len(conflicts),
               'images_with_conflicting_labels': int(images.label_conflict.sum()),
               'eligible_before_manual_review': int(images.eligible.sum()),
               'near_duplicate_candidate_pairs': len(near), 'dhash_distance_threshold': near_distance,
               'age_metadata_present_in_label_csv': any('age' == c.lower() for c in labels.columns),
               'patient_id_present_in_label_csv': any(c.lower() in {'patient', 'patient_id'} for c in labels.columns),
               'limitations': ['No claim of patient-level separation without patient metadata.',
                              'dHash candidates require manual review; candidate absence does not prove independence.',
                              'Conflicting exact-image labels are quarantined from splits, not corrected.',
                              'No expert assessment of clinical image quality or label correctness was performed.'],
               'environment': {'python': sys.version, 'platform': platform.platform(),
                               'pandas': pd.__version__, 'numpy': np.__version__, 'Pillow': pillow_version}}
    save_json(output_dir / 'audit_summary.json', summary)
    make_plots(labels, images, output_dir)
    print(summary, flush=True)
    return summary

In [ ]:
audit_summary = None
if MODE == 'build':
    audit_summary = run_audit(LABELS_PATH, RAW_IMAGES, AUDIT_DIR,
                              workers=WORKERS, near_distance=NEAR_DISTANCE)
elif (AUDIT_DIR / 'audit_summary.json').is_file():
    audit_summary = json.loads((AUDIT_DIR / 'audit_summary.json').read_text(encoding='utf-8'))
    if audit_summary['labels_sha256'] != file_hash(LABELS_PATH):
        raise ValueError('The recorded audit belongs to a different label file.')
    print('Loaded the recorded audit; original images were not re-audited.')
else:
    print('This prepared handoff has no audit summary. Raw-image audit was not rerun.')

if audit_summary is not None:
    keys = ['label_rows', 'readable_images', 'missing_images', 'unreadable_images',
            'exact_duplicate_groups', 'images_in_exact_duplicate_groups',
            'conflicting_label_groups', 'images_with_conflicting_labels',
            'eligible_before_manual_review', 'near_duplicate_candidate_pairs']
    display(pd.DataFrame({'audit_measure': keys, 'value': [audit_summary[k] for k in keys]}))

## 4. Class distribution and audit findings

The table and chart below use the label file on **this computer**. Compare them
with the separately recorded APTOS results in section 8. Class imbalance motivates
reporting per-grade metrics rather than relying on overall accuracy alone.

In [ ]:
counts = labels.diagnosis.value_counts().reindex(range(5), fill_value=0)
class_table = pd.DataFrame({
    'grade': range(5), 'class': [CLASS_NAMES[k] for k in range(5)],
    'count': counts.to_numpy(), 'percent': counts.to_numpy() / len(labels) * 100,
})
display(class_table.round({'percent': 2}))
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(class_table['class'], class_table['count'], color='#237a91')
ax.bar_label(bars, padding=3)
ax.set_title('Original label distribution in the current dataset')
ax.set_ylabel('Number of images')
ax.set_ylim(0, max(counts) * 1.15)
fig.tight_layout()
display(fig)
plt.close(fig)

## 5. Group-aware provisional splitting

Exact duplicate groups remain in one split. Optional documented related-image
groups are combined with exact matches. A fixed-seed search selects approximately
70% training, 15% validation and 15% test, balancing size and class proportions;
it never selects a split using model performance. Same-grade duplicates are
retained within a split, so row counts are not counts of independent patients.

Review similar images and resolve the final grouping policy before freezing a
split for final evaluation. Do not use the reserved test set to select methods.

In [ ]:
def choose_holdout(frame, fraction, seed):
    """Select the closest class-balanced group split from deterministic candidates."""
    total = np.bincount(frame.diagnosis, minlength=5)
    if (total == 0).any():
        raise ValueError('All five grades must be available before splitting.')
    sampler = GroupShuffleSplit(n_splits=256, test_size=fraction, random_state=seed)
    best = None
    for train, holdout in sampler.split(frame, groups=frame.group_id):
        hold_counts = np.bincount(frame.iloc[holdout].diagnosis, minlength=5)
        train_counts = total - hold_counts
        if (hold_counts == 0).any() or (train_counts == 0).any():
            continue
        score = np.mean(np.abs(hold_counts / total - fraction))
        score += abs(len(holdout) / len(frame) - fraction)
        if best is None or score < best[0]:
            best = score, train, holdout
    if best is None:
        raise ValueError('Unable to preserve every grade across splits. Review group/sample counts.')
    return frame.iloc[best[1]].copy(), frame.iloc[best[2]].copy()


def attach_groups(frame, reviewed_groups=None):
    """Union exact pixel matches and any user-supplied reviewed related-image groups."""
    parent = {i: i for i in frame.id_code}
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    for _, rows in frame.groupby('pixel_sha256'):
        ids = list(rows.id_code)
        for image_id in ids[1:]:
            union(ids[0], image_id)
    if reviewed_groups is not None:
        reviewed = pd.read_csv(reviewed_groups, dtype=str)
        if not {'id_code', 'reviewed_group'} <= set(reviewed.columns):
            raise ValueError('Reviewed group CSV must contain id_code and reviewed_group.')
        if reviewed[['id_code', 'reviewed_group']].isna().any().any() or reviewed.id_code.duplicated().any():
            raise ValueError('Reviewed group entries must be non-empty and IDs unique.')
        if not set(reviewed.id_code) <= set(frame.id_code):
            raise ValueError('Reviewed group CSV references unknown/excluded IDs.')
        for _, rows in reviewed.groupby('reviewed_group'):
            ids = list(rows.id_code)
            for image_id in ids[1:]:
                union(ids[0], image_id)
    result = frame.copy()
    result['group_id'] = [find(i) for i in result.id_code]
    return result


def run_split(inventory, destination, seed=2112, reviewed_groups=None):
    frame = pd.read_csv(inventory, dtype={'id_code': str, 'pixel_sha256': str})
    # Conflicting identical-image labels stay out until a documented review resolves them.
    eligible = frame[frame.status.eq('ok') & ~frame.label_conflict].copy()
    eligible = eligible.sort_values('id_code').reset_index(drop=True)
    eligible = attach_groups(eligible, reviewed_groups)
    development, test = choose_holdout(eligible, .15, seed)
    train, validation = choose_holdout(development, .15 / .85, seed + 1)
    pieces = []
    for name, part in [('train', train), ('validation', validation), ('test', test)]:
        part['split'] = name
        pieces.append(part)
    manifest = pd.concat(pieces, ignore_index=True).sort_values('id_code')
    if manifest.groupby('group_id')['split'].nunique().max() != 1:
        raise AssertionError('Related-image group crosses splits.')
    if manifest.groupby('pixel_sha256')['split'].nunique().max() != 1:
        raise AssertionError('Exact pixel duplicate crosses splits.')
    if manifest.id_code.duplicated().any() or len(manifest) != len(eligible):
        raise AssertionError('Coverage or uniqueness check failed.')
    destination.mkdir(parents=True, exist_ok=True)
    columns = ['id_code', 'diagnosis', 'split', 'group_id', 'pixel_sha256']
    manifest[columns].to_csv(destination / 'split_manifest.csv', index=False)
    for name in ['train', 'validation', 'test']:
        manifest.loc[manifest.split.eq(name), columns].to_csv(destination / f'{name}.csv', index=False)
    excluded = frame.loc[~frame.id_code.isin(manifest.id_code), ['id_code', 'diagnosis', 'status', 'label_conflict']]
    excluded.to_csv(destination / 'excluded_records.csv', index=False)
    counts = pd.crosstab(manifest['split'], manifest.diagnosis).reindex(['train', 'validation', 'test'])
    counts.to_csv(destination / 'split_class_counts.csv')
    summary = {'status': 'PROVISIONAL: review similar images and any available patient metadata before final evaluation',
               'seed': seed, 'target_proportions': {'train': .70, 'validation': .15, 'test': .15},
               'actual_counts': manifest.split.value_counts().to_dict(),
               'class_counts': {split: {str(k): int(v) for k, v in row.items()} for split, row in counts.iterrows()},
               'eligible_images': len(manifest), 'excluded_images': len(excluded),
               'groups': int(manifest.group_id.nunique()), 'exact_pixel_overlap_across_splits': 0,
               'inventory_sha256': file_hash(inventory),
               'reviewed_groups_sha256': file_hash(reviewed_groups) if reviewed_groups else None,
               'notes': ['GroupShuffleSplit candidates selected by size and per-class proportional error, not model scores.',
                         'Same exact pixels remain in one split; duplicates are retained, not counted as independent patients.',
                         'No random augmentations or resampling applied to validation/test.',
                         'Freeze reviewed splits before model selection; do not tune on test results.']}
    save_json(destination / 'split_summary.json', summary)
    print(counts.to_string())
    print(f'Wrote {len(manifest)} eligible rows; {len(excluded)} excluded. Splits remain provisional.')
    return manifest

In [ ]:
if MODE == 'build':
    run_split(AUDIT_DIR / 'image_inventory.csv', SPLIT_DIR,
              seed=SEED, reviewed_groups=REVIEWED_GROUPS)

split_manifest = pd.read_csv(SPLIT_DIR / 'split_manifest.csv', dtype={'id_code': str})
split_summary = json.loads((SPLIT_DIR / 'split_summary.json').read_text(encoding='utf-8'))
if split_manifest.id_code.duplicated().any():
    raise ValueError('The split manifest repeats image IDs.')
if not set(split_manifest.id_code) <= set(labels.id_code):
    raise ValueError('The split manifest contains unknown IDs.')
label_map = labels.set_index('id_code').diagnosis
if not split_manifest.diagnosis.eq(split_manifest.id_code.map(label_map)).all():
    raise ValueError('Split labels do not match the original labels.')
for key in ('group_id', 'pixel_sha256'):
    if split_manifest.groupby(key)['split'].nunique().max() != 1:
        raise ValueError(f'{key} crosses split boundaries.')

split_counts = pd.crosstab(split_manifest['split'], split_manifest.diagnosis).reindex(
    index=['train', 'validation', 'test'], columns=range(5), fill_value=0)
split_counts['total'] = split_counts.sum(axis=1)
display(split_counts)
print('Eligible:', len(split_manifest), '| Excluded:', len(labels) - len(split_manifest))
print('Split SHA-256:', file_hash(SPLIT_DIR / 'split_manifest.csv'))
print(split_summary['status'])

## 6. Deterministic image preparation

Baseline: RGB, 224 x 224, aspect-preserving LANCZOS resize and black padding.
Automatic black-border cropping is **off**. No model normalization or augmentation
is stored in these PNGs. B applies the chosen model's normalization exactly once.
224 pixels is a computational baseline; preservation of small retinal lesions is
not established. If B needs another resolution, regenerate from original images
and agree on a new handoff contract.

In [ ]:
def prepare_image(image, size=224, crop_black=False, threshold=7):
    rgb = image.convert('RGB')
    bounds = (0, 0, rgb.width, rgb.height)
    if crop_black:
        # Bounding box of non-dark pixels; optional experiment, not a clinically validated segmentation.
        mask = Image.fromarray((np.asarray(rgb).max(axis=2) > threshold).astype('uint8') * 255)
        box = mask.getbbox()
        if box is not None:
            bounds = box
            rgb = rgb.crop(box)
    # Preserve aspect ratio; do not stretch a circular retina into an ellipse.
    prepared = ImageOps.pad(rgb, (size, size), method=Image.Resampling.LANCZOS, color=(0, 0, 0))
    return prepared, bounds


def run_preprocess(manifest_path, image_dir, output_dir, size=224, crop_black=False,
                   threshold=7, workers=4, limit=None):
    if size < 16 or not 0 <= threshold <= 255:
        raise ValueError('Invalid image size or black-border threshold.')
    if output_dir.resolve() == image_dir.resolve() or image_dir.resolve() in output_dir.resolve().parents:
        raise ValueError('Output must be outside the original image directory.')
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError('Output already contains files. Choose a new versioned directory.')
    frame = pd.read_csv(manifest_path, dtype={'id_code': str})
    if not {'id_code', 'diagnosis', 'split'} <= set(frame.columns):
        raise ValueError('Manifest is missing required columns.')
    if not frame.id_code.str.fullmatch(r'[A-Za-z0-9_-]+').all() or frame.id_code.duplicated().any():
        raise ValueError('Unsafe or duplicate ID in manifest.')
    if not frame.diagnosis.isin(range(5)).all() or not frame.split.isin(['train', 'validation', 'test']).all():
        raise ValueError('Unexpected grade or split.')
    if limit is not None:
        if limit < 1:
            raise ValueError('Limit must be positive.')
        frame = frame.head(limit)
    output_dir.mkdir(parents=True, exist_ok=True)
    def convert(row):
        path = image_dir / f'{row.id_code}.png'
        with Image.open(path) as source:
            prepared, box = prepare_image(source, size, crop_black, threshold)
        target = output_dir / f'{row.id_code}.png'
        prepared.save(target)
        return {'id_code': row.id_code, 'diagnosis': int(row.diagnosis), 'split': row.split,
                'relative_path': target.name, 'crop_box': ','.join(map(str, box))}
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for number, item in enumerate(pool.map(convert, frame.itertuples()), 1):
            results.append(item)
            if number % 500 == 0 or number == len(frame):
                print(f'Prepared {number}/{len(frame)} images', flush=True)
    pd.DataFrame(results).to_csv(output_dir / 'processed_manifest.csv', index=False)
    config = {'status': 'EXPERIMENTAL BASELINE - confirm model input requirements with teammate',
              'size': size, 'mode': 'RGB', 'format': 'lossless PNG, uint8 0..255',
              'resize': 'aspect-preserving LANCZOS resize and black padding',
              'crop_black': crop_black, 'black_threshold': threshold, 'rows': len(results),
              'manifest_sha256': file_hash(manifest_path), 'augmentation': 'none',
              'normalization': 'none; apply model-specific normalization in training/inference code',
              'notes': ['224 is a compute-friendly baseline, not a validated optimal size for retinal lesions.',
                        'No clinical claim about preservation of small lesions; review full-resolution examples.',
                        'Use the same deterministic transform for train, validation and test.']}
    save_json(output_dir / 'preprocessing_config.json', config)
    # One local before/after example for each available grade.
    sample_rows = frame.groupby('diagnosis').head(1)
    sheet = Image.new('RGB', (440, len(sample_rows) * 250), 'white')
    draw = ImageDraw.Draw(sheet)
    for index, row in enumerate(sample_rows.itertuples()):
        for col, source_path in enumerate([image_dir / f'{row.id_code}.png', output_dir / f'{row.id_code}.png']):
            with Image.open(source_path) as source:
                preview = ImageOps.pad(source.convert('RGB'), (210, 210))
            sheet.paste(preview, (col * 220, index * 250))
        draw.text((5, index * 250 + 215), f'Grade {row.diagnosis} | original (display scaled) -> prepared', fill='black')
        draw.text((5, index * 250 + 230), row.id_code, fill='black')
    if len(sample_rows):
        sheet.save(output_dir / 'PRIVATE_before_after.jpg', quality=90)
    print(f'Output: {output_dir}')

In [ ]:
if MODE == 'build':
    run_preprocess(SPLIT_DIR / 'split_manifest.csv', RAW_IMAGES, PREPARED_DIR,
                   size=224, crop_black=False, workers=WORKERS)

preprocessing_config = json.loads((PREPARED_DIR / 'preprocessing_config.json').read_text(encoding='utf-8'))
if preprocessing_config['manifest_sha256'] != file_hash(SPLIT_DIR / 'split_manifest.csv'):
    raise ValueError('Prepared images refer to a different split manifest.')
display(pd.DataFrame([
    {'setting': key, 'value': preprocessing_config[key]}
    for key in ['size', 'mode', 'resize', 'crop_black', 'rows', 'augmentation', 'normalization']
]))

## 7. Verified loading and handoff to B

These complete loader functions join records by `id_code`, validate labels and
groups, and require the expected prepared-image paths. `load_split('test', ... )`
is blocked by default. The verifier checks the metadata and every prepared image
against the handoff fingerprints. Its success is a software/data-consistency
result, not evidence of medical accuracy or completed manual review.

In [ ]:
def load_manifest(root=ROOT):
    """Join labels and image paths by ID, never by directory order."""
    root = Path(root)
    split_path = root / 'artifacts/splits/split_manifest.csv'
    image_dir = root / 'data/processed_224_v1'
    manifest = pd.read_csv(split_path, dtype={'id_code': str})
    prepared = pd.read_csv(image_dir / 'processed_manifest.csv', dtype={'id_code': str})
    if not {'id_code', 'diagnosis', 'split', 'group_id', 'pixel_sha256'} <= set(manifest.columns):
        raise ValueError('Split manifest lacks required columns.')
    if not {'id_code', 'diagnosis', 'split', 'relative_path'} <= set(prepared.columns):
        raise ValueError('Prepared manifest lacks required columns.')
    for frame in (manifest, prepared):
        if frame.id_code.isna().any() or not frame.id_code.str.fullmatch(r'[A-Za-z0-9_-]+').all():
            raise ValueError('Missing or unsafe image ID.')
        if frame.id_code.duplicated().any() or not frame.diagnosis.isin(range(5)).all():
            raise ValueError('Repeated ID or invalid diagnosis.')
        if not frame.split.isin(['train', 'validation', 'test']).all():
            raise ValueError('Invalid split name.')
    if manifest[['group_id', 'pixel_sha256']].isna().any().any():
        raise ValueError('Missing duplicate group information.')
    if set(manifest.id_code) != set(prepared.id_code):
        raise ValueError('Prepared and split manifests contain different IDs.')
    for key in ('group_id', 'pixel_sha256'):
        if manifest.groupby(key)['split'].nunique().max() != 1:
            raise ValueError(f'{key} crosses split boundaries.')
    merged = manifest.merge(prepared[['id_code', 'diagnosis', 'split', 'relative_path']],
                            on='id_code', validate='one_to_one', suffixes=('', '_prepared'))
    if not merged.diagnosis.eq(merged.diagnosis_prepared).all() or not merged.split.eq(merged.split_prepared).all():
        raise ValueError('Prepared labels/splits disagree with the split manifest.')
    if not merged.relative_path.eq(merged.id_code + '.png').all():
        raise ValueError('Unexpected image filename; paths must be ID.png.')
    config = json.loads((image_dir / 'preprocessing_config.json').read_text(encoding='utf-8'))
    if config['manifest_sha256'] != file_hash(split_path):
        raise ValueError('Prepared images were made with a different split manifest.')
    if config['rows'] != len(merged) or config['size'] != 224 or config['mode'] != 'RGB':
        raise ValueError('Unexpected preprocessing configuration for this handoff.')
    merged['image_path'] = [str(image_dir / name) for name in merged.relative_path]
    if not all(Path(path).is_file() for path in merged.image_path):
        raise FileNotFoundError('Some prepared images are missing; extract the data pack first.')
    return merged.drop(columns=['diagnosis_prepared', 'split_prepared'])


def load_split(split='train', root=ROOT, allow_test=False):
    if split not in {'train', 'validation', 'test'}:
        raise ValueError('Use train, validation or test.')
    if split == 'test' and not allow_test:
        raise ValueError('Test is reserved. Review/freeze the split before final evaluation; do not tune on test.')
    manifest = load_manifest(root)
    return manifest[manifest.split.eq(split)].reset_index(drop=True)


def read_example(row):
    """Return HWC RGB uint8 pixels and integer grade; normalization is B's responsibility."""
    with Image.open(row.image_path) as image:
        if image.mode != 'RGB' or image.size != (224, 224):
            raise ValueError('Expected prepared 224x224 RGB image.')
        pixels = np.array(image)
    return pixels, int(row.diagnosis)


def verify(root=ROOT):
    root = Path(root)
    contract = json.loads((root / 'docs/handoff_contract.json').read_text(encoding='utf-8'))
    for relative, expected in contract['metadata_sha256'].items():
        path = (root / relative).resolve()
        if root.resolve() not in path.parents:
            raise ValueError('Contract path is outside the project.')
        if file_hash(path) != expected:
            raise ValueError(f'Changed/mismatched metadata: {relative}')
    frame = load_manifest(root)
    if frame.split.value_counts().to_dict() != contract['split_counts']:
        raise ValueError('Split counts do not match the handoff version.')
    checks = pd.read_csv(root / 'artifacts/handoff/image_checksums.csv', dtype=str)
    if checks.id_code.duplicated().any() or set(checks.id_code) != set(frame.id_code):
        raise ValueError('Image checksum inventory has different IDs.')
    hashes = checks.set_index('id_code').sha256.to_dict()
    def check(row):
        if file_hash(row.image_path) != hashes[row.id_code]:
            raise ValueError(f'Changed prepared image: {row.id_code}')
        pixels, grade = read_example(row)
        if pixels.dtype != np.uint8 or grade not in range(5):
            raise ValueError('Unexpected pixel type or grade.')
    with ThreadPoolExecutor(max_workers=4) as pool:
        list(pool.map(check, frame.itertuples()))
    result = {'version': contract['version'], 'status': contract['status'],
              'verified_images': len(frame), 'split_counts': contract['split_counts'],
              'exact_duplicate_overlap': 0, 'manual_similarity_review_complete': False}
    print(json.dumps(result, indent=2))
    print('PASS: file hashes, image decoding, IDs, grades, split assignments and duplicate groups.')
    print('This verifies the handoff files, not clinical labels or patient independence.')
    return result

### Create fingerprints for a new build, or check the existing private handoff

In build mode, this cell copies the authorised label CSV into the new local run,
records image checksums and creates a **provisional** contract. Nothing is uploaded
by running this Notebook. The complete new handoff root is printed below.

In [ ]:
if MODE == 'build':
    (WORK_ROOT / 'data').mkdir(parents=True, exist_ok=True)
    shutil.copy2(LABELS_PATH, WORK_ROOT / 'data/train.csv')
    handoff_images = load_manifest(WORK_ROOT)
    checksum_path = WORK_ROOT / 'artifacts/handoff/image_checksums.csv'
    checksum_path.parent.mkdir(parents=True, exist_ok=True)
    checksum_rows = [
        {'id_code': row.id_code, 'sha256': file_hash(row.image_path)}
        for row in handoff_images.sort_values('id_code').itertuples()
    ]
    pd.DataFrame(checksum_rows).to_csv(checksum_path, index=False)
    metadata_paths = [
        'data/train.csv', 'artifacts/splits/split_manifest.csv',
        'artifacts/splits/split_summary.json', 'artifacts/splits/excluded_records.csv',
        'data/processed_224_v1/processed_manifest.csv',
        'data/processed_224_v1/preprocessing_config.json',
        'artifacts/handoff/image_checksums.csv',
    ]
    contract = {
        'version': RUN_NAME,
        'status': 'PROVISIONAL: manual review and team input agreement remain required',
        'classes': CLASS_NAMES,
        'split_counts': handoff_images.split.value_counts().to_dict(),
        'image_shape_hwc': [224, 224, 3], 'pixel_dtype': 'uint8', 'pixel_range': [0, 255],
        'model_normalization': None, 'augmentation': None,
        'metadata_sha256': {path: file_hash(WORK_ROOT / path) for path in metadata_paths},
        'manual_similarity_review_complete': False, 'patient_independence_verified': False,
        'model_input_requirements_confirmed_with_B': False,
    }
    save_json(WORK_ROOT / 'docs/handoff_contract.json', contract)

contract_path = WORK_ROOT / 'docs/handoff_contract.json'
if not contract_path.is_file():
    raise FileNotFoundError('The prepared handoff contract is missing. Obtain the matching private '
                            'handoff, or create a new version with build mode.')
verification_result = verify(WORK_ROOT)
print('Verified local handoff root:', WORK_ROOT)

In [ ]:
train = load_split('train', root=WORK_ROOT)
validation = load_split('validation', root=WORK_ROOT)
pixels, label = read_example(train.iloc[0])
print('Training records:', len(train), '| Validation records:', len(validation))
print('Example array:', pixels.shape, pixels.dtype, '| Grade:', label)
print('RGB, HWC, integer pixels 0..255. Apply model-specific transforms exactly once.')
print('The reserved test split has not been loaded for model evaluation.')

# Optional LOCAL preview. Keep this off in the publicly shared Notebook.
if SHOW_PRIVATE_PREVIEW:
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(pixels)
    ax.set_title(f'Training example: {CLASS_NAMES[label]}')
    ax.axis('off')
    display(fig)
    plt.close(fig)

### B's training interface

Use `train` and `validation` above as the inputs to your framework's Dataset or
DataLoader. `read_example(row)` returns an HWC RGB `uint8` array and an integer
grade. Convert channel order and floating-point scale as required by the selected
weights. Keep augmentation/rebalancing training-only. Do not split again randomly.

When sharing a new build, keep the `WORK_ROOT` folder structure intact and provide
this Notebook. On B's computer, set `PROJECT_ROOT` to that extracted root and use
`MODE = 'inspect'`. B does not need the full-resolution originals to load prepared
images. Rebuilding at another resolution does require the originals.

The `.py` loading interface remains available in the repository for integration
into training scripts. This Notebook is self-contained for interactive work.

## 8. Recorded data analysis and report contribution

The following are **recorded results from A's 7 October 2026 audit**, used in the
13 October provisional handoff. They are not results generated merely by opening
this Notebook, and they are not model performance measurements. Use the live
tables above if your input data or reviewed split changes.

### Source, labels and distribution

Source: [APTOS 2019 Blindness Detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection/data).
The original labelled training set contains 3,662 retinal images. `id_code` links
each label record to its PNG; `diagnosis` is the original grade, 0-4.

| Grade | Meaning | Original count |
|---|---|---:|
| 0 | No DR | 1,805 |
| 1 | Mild | 370 |
| 2 | Moderate | 999 |
| 3 | Severe | 193 |
| 4 | Proliferative DR | 295 |

The uneven distribution means overall accuracy alone may conceal weaker minority
grade performance. Metric selection and any balancing strategy need team agreement.

### Integrity, repetition and exclusions

All 3,662 files decoded successfully, with no missing or extra PNGs in the audited
directory. There were 123 exact-pixel duplicate groups containing 251 records.
Thirty groups had conflicting grades, accounting for 62 records. These records
were provisionally excluded without assigning guessed replacement grades.

The remaining 3,600 records represent 3,504 exact-image groups. Same-grade duplicate
records were retained within a single split. The similarity search produced 4,096
candidate pairs; these are not 4,096 confirmed duplicates or patient identities.

### Provisional split

| Split | Grade 0 | Grade 1 | Grade 2 | Grade 3 | Grade 4 | Total |
|---|---:|---:|---:|---:|---:|---:|
| Training | 1,258 | 253 | 684 | 127 | 199 | 2,521 |
| Validation | 270 | 54 | 148 | 26 | 40 | 538 |
| Test | 273 | 53 | 144 | 27 | 44 | 541 |

No exact-pixel group crossed these splits. Similar-image review is incomplete,
and the available labels contain neither patient IDs nor age information.
Patient-level separation, age-subgroup performance and demographic fairness are
therefore not established by this audit.

### Preprocessing and interpretation

The prepared baseline uses 224 x 224 RGB images, aspect-preserving LANCZOS resizing
and black padding. Black cropping is off; normalization and random augmentation
are left to B's training pipeline. These choices reduce input variation and
computation, but do not demonstrate preservation of small retinal lesions.

### Changes from the proposal

| Topic | Original plan | Current data contribution | Reason / unresolved issue |
|---|---|---|---|
| Prediction target | Binary grouping | Original five grades | Retains severity detail; model and metric agreement still required |
| Preprocessing | Resizing, normalization and black-border handling proposed | 224 RGB resize/padding; crop off; normalization deferred | Avoid unvalidated cropping and match normalization to selected weights |
| Data split | To be prepared | Fixed-seed exact-group split | Reduces known duplicate overlap; does not prove patient independence |

### Contribution to the group report

Use this material for Dataset Description, Data Quality, Preprocessing, Splitting,
Data Findings and Data Limitations. Add the student's own explanations, final
review decisions and suitable references. Background/related work still needs
completion. The group Final Report also needs B's models and experiments, C's
evaluation and deployment discussion, recommendations, and the full AI appendix.
The previous PDF/DOCX data reports remain supplementary exports of the recorded
analysis; they are not a completed group Final Report.

## 9. Acceptance checklist, limitations and AI-use record

### A and B acceptance checklist

- [ ] B has authorised dataset access and the matching private handoff.
- [ ] B selects the correct Python kernel and runs this Notebook locally.
- [ ] File verification succeeds and image/label loading works on B's computer.
- [ ] The team agrees on the five-grade target, model, weights and evaluation metrics.
- [ ] B records resolution, channel order, scaling, normalization and augmentation.
- [ ] A documents duplicate/conflict decisions and the limitations of related-image review.
- [ ] The reviewed split is frozen before final model evaluation.
- [ ] Everyone records the same split fingerprint and handoff version.

The recorded baseline is ready for pipeline integration and preliminary
train/validation experiments. It is not a frozen final-evaluation dataset. Do not
tick the checklist merely because the software has run successfully on A's computer.

### Public sharing and local outputs

Images and per-image records remain local and subject to dataset terms. Use an
authorised team channel for the private handoff. Clear **all saved outputs** before
publishing a rerun Notebook, particularly if private preview was enabled. In VS Code
use **Clear All Outputs** and save. The repository command-line clearing utility is
also available in `src/clear_notebook_outputs.py`.

### AI assistance and outstanding writing

Codex assisted with the scripts, this Notebook, analysis wording and handoff
materials. Students must understand, review and revise the work. Preserve relevant
prompts, outputs, changes and verification evidence for the course AI-use appendix.
This disclosure does not replace that appendix. No medical expert label review,
patient-level independence, demographic performance, trained model result or
reduction in clinical workload is claimed.

Source documentation: [APTOS data](https://www.kaggle.com/competitions/aptos2019-blindness-detection/data)
and [competition rules](https://www.kaggle.com/competitions/aptos2019-blindness-detection/rules).